In [1]:
# RegMap – Sentence‑BERT Fine‑Tuning for NIST‑to‑HIPAA Mapping

In [13]:
import pandas as pd
import numpy as np
import random
import json
import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader
import torch

# ---------- Paths ----------
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path('../models/regmap-embedder')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibility ----------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

In [14]:
# Reload raw crosswalk
crosswalk_raw = pd.read_csv(DATA_RAW / 'nist_800_53_rev5_hipaa_crosswalk_full.csv')

# Clean column names (remove newlines / extra spaces)
crosswalk_raw.columns = (
    crosswalk_raw.columns
    .str.replace('\n', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Rename to our standard names
crosswalk = crosswalk_raw.rename(columns={
    'Focal Document Element': 'nist_control_id',
    'Focal Document Element Description': 'nist_text',
    'Reference Document Element': 'hipaa_citation',
    'Reference Document Element Description': 'hipaa_text'
})

# Keep only the columns we need
crosswalk = crosswalk[['nist_control_id', 'nist_text', 'hipaa_citation', 'hipaa_text']]

# Replace empty strings with NaN and drop any row missing text
crosswalk.replace('', np.nan, inplace=True)
crosswalk.dropna(subset=['nist_text', 'hipaa_text', 'hipaa_citation'], inplace=True)

# Strip whitespace from all text columns
for col in ['nist_text', 'hipaa_text', 'hipaa_citation']:
    crosswalk[col] = crosswalk[col].astype(str).str.strip()

# Drop exact duplicate mappings (same control + same citation)
crosswalk = crosswalk.drop_duplicates(subset=['nist_control_id', 'hipaa_citation']).copy()

# All remaining rows are positive (true) mappings
crosswalk['label'] = 1

print(f"Total positive pairs: {len(crosswalk)}")
crosswalk.head()

Total positive pairs: 222


,nist_control_id,nist_text,hipaa_citation,hipaa_text,label
0,AC-1,"Policy and Procedures: Develop, document, and ...",164.308(a)(3),Workforce Security: Implement policies and pro...,1
1,AC-1,"Policy and Procedures: Develop, document, and ...",164.308(a)(4),Information Access Management: Implement polic...,1
2,AC-1,"Policy and Procedures: Develop, document, and ...",164.312(a),Access Control: Implement technical policies a...,1
3,AC-1,"Policy and Procedures: Develop, document, and ...",164.316(b)(2)(ii),Availability (R): Make documentation available...,1
4,AC-1,"Policy and Procedures: Develop, document, and ...",164.316(b)(2)(iii),Updates (R): Review documentation periodically...,1


In [15]:
# Stratification not needed (all labels are 1), just simple random splits
train_val, test = train_test_split(
    crosswalk,
    test_size=0.15,
    random_state=SEED
)
train, val = train_test_split(
    train_val,
    test_size=0.15,   # ~12.75% of total for val
    random_state=SEED
)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Train: 159 | Val: 29 | Test: 34


In [16]:
def df_to_examples(df):
    examples = []
    for _, row in df.iterrows():
        # MultipleNegativesRankingLoss uses implicit positives:
        # the pair is the positive, all other pairs in batch are negatives.
        examples.append(InputExample(texts=[row['nist_text'], row['hipaa_text']]))
    return examples

train_examples = df_to_examples(train)
val_examples   = df_to_examples(val)
test_examples  = df_to_examples(test)

print(f"Train examples: {len(train_examples)}")
print(f"Val examples:   {len(val_examples)}")
print(f"Test examples:  {len(test_examples)}")

Train examples: 159
Val examples:   29
Test examples:  34


In [17]:
model = SentenceTransformer('all-MiniLM-L6-v2')
train_loss = MultipleNegativesRankingLoss(model)
print("Model loaded with MultipleNegativesRankingLoss.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded with MultipleNegativesRankingLoss.


In [18]:
# Build corpus from ALL unique HIPAA texts in the full crosswalk (not just val split)
# Using the full corpus makes retrieval non-trivial: 60 provisions to rank against.
all_hipaa_texts = crosswalk['hipaa_text'].unique()
corpus = {i: text for i, text in enumerate(all_hipaa_texts)}

# For each validation query (NIST text), specify the relevant HIPAA citation(s)
queries = {}
relevant_docs = {}
qid = 0
for _, row in val.iterrows():
    query_text = row['nist_text']
    true_text = row['hipaa_text']
    true_id = None
    for idx, ctext in corpus.items():
        if ctext == true_text:
            true_id = idx
            break
    if true_id is not None:
        queries[qid] = query_text
        relevant_docs[qid] = {true_id}
        qid += 1

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="val-hipaa",
    show_progress_bar=True
)
print(f"Validation queries: {len(queries)}, corpus size: {len(corpus)}")

Validation queries: 29, corpus size: 60


In [19]:
# import sys
# !{sys.executable} -m pip install --upgrade "accelerate>=1.1.0"


In [20]:
BATCH_SIZE = 16          # larger batch helps MultipleNegativesRankingLoss
EPOCHS = 10
WARMUP_STEPS = int(len(train_examples) / BATCH_SIZE * 0.1)

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# evaluation_steps=10 fires roughly once per epoch (~10 steps/epoch with ~161 train pairs)
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    evaluator=ir_evaluator,
    evaluation_steps=10,
    output_path=str(MODEL_DIR),
    save_best_model=True,
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\stamp\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Val-hipaa Cosine Accuracy@1,Val-hipaa Cosine Accuracy@3,Val-hipaa Cosine Accuracy@5,Val-hipaa Cosine Accuracy@10,Val-hipaa Cosine Precision@1,Val-hipaa Cosine Precision@3,Val-hipaa Cosine Precision@5,Val-hipaa Cosine Precision@10,Val-hipaa Cosine Recall@1,Val-hipaa Cosine Recall@3,Val-hipaa Cosine Recall@5,Val-hipaa Cosine Recall@10,Val-hipaa Cosine Ndcg@10,Val-hipaa Cosine Mrr@10,Val-hipaa Cosine Map@100
10,No log,No log,0.310345,0.620690,0.655172,0.793103,0.310345,0.206897,0.131034,0.079310,0.310345,0.620690,0.655172,0.793103,0.544079,0.464833,0.475321
20,No log,No log,0.310345,0.586207,0.655172,0.862069,0.310345,0.195402,0.131034,0.086207,0.310345,0.586207,0.655172,0.862069,0.570745,0.480364,0.486007
30,No log,No log,0.310345,0.586207,0.724138,0.862069,0.310345,0.195402,0.144828,0.086207,0.310345,0.586207,0.724138,0.862069,0.572060,0.481500,0.487598
40,No log,No log,0.241379,0.620690,0.689655,0.827586,0.241379,0.206897,0.137931,0.082759,0.241379,0.620690,0.689655,0.827586,0.535743,0.442337,0.451169
50,No log,No log,0.275862,0.620690,0.724138,0.827586,0.275862,0.206897,0.144828,0.082759,0.275862,0.620690,0.724138,0.827586,0.551948,0.463314,0.472767
60,No log,No log,0.275862,0.586207,0.758621,0.862069,0.275862,0.195402,0.151724,0.086207,0.275862,0.586207,0.758621,0.862069,0.562709,0.467378,0.474511
70,No log,No log,0.275862,0.586207,0.758621,0.862069,0.275862,0.195402,0.151724,0.086207,0.275862,0.586207,0.758621,0.862069,0.563122,0.467761,0.475319
80,No log,No log,0.275862,0.620690,0.758621,0.896552,0.275862,0.206897,0.151724,0.089655,0.275862,0.620690,0.758621,0.896552,0.575067,0.473700,0.478147
90,No log,No log,0.275862,0.620690,0.758621,0.896552,0.275862,0.206897,0.151724,0.089655,0.275862,0.620690,0.758621,0.896552,0.575067,0.473700,0.478171
100,No log,No log,0.275862,0.620690,0.758621,0.896552,0.275862,0.206897,0.151724,0.089655,0.275862,0.620690,0.758621,0.896552,0.575067,0.473700,0.478171


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

...

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]


In [21]:
# Load the best model saved during training
best_model = SentenceTransformer(str(MODEL_DIR))

# Build test corpus from ALL unique HIPAA provisions (60 total) — not just the test split.
# This makes Recall@K meaningful: the model must retrieve the correct provision
# from a full 60-item pool, not a trivial 6-item one.
all_hipaa_texts = crosswalk['hipaa_text'].unique()
test_corpus = {i: text for i, text in enumerate(all_hipaa_texts)}
test_corpus_list = list(test_corpus.values())
corpus_embeddings = best_model.encode(test_corpus_list, convert_to_tensor=True)

recall_1 = recall_3 = recall_5 = mrr = 0
total_queries = 0

for _, row in test.iterrows():
    query_text = row['nist_text']
    true_text = row['hipaa_text']

    true_id = None
    for idx, ctext in test_corpus.items():
        if ctext == true_text:
            true_id = idx
            break
    if true_id is None:
        continue

    query_embed = best_model.encode(query_text, convert_to_tensor=True)
    cos_scores = torch.nn.functional.cosine_similarity(query_embed, corpus_embeddings)
    sorted_indices = torch.argsort(cos_scores, descending=True)
    rank = (sorted_indices == true_id).nonzero(as_tuple=True)[0].item() + 1

    if rank <= 1: recall_1 += 1
    if rank <= 3: recall_3 += 1
    if rank <= 5: recall_5 += 1
    mrr += 1 / rank
    total_queries += 1

recall_1 /= total_queries
recall_3 /= total_queries
recall_5 /= total_queries
mrr /= total_queries

print(f"Test queries evaluated: {total_queries}")
print(f"Corpus size (HIPAA provisions): {len(test_corpus)}")
print(f"Recall@1: {recall_1:.4f}")
print(f"Recall@3: {recall_3:.4f}")
print(f"Recall@5: {recall_5:.4f}")
print(f"MRR:      {mrr:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Test queries evaluated: 34
Corpus size (HIPAA provisions): 60
Recall@1: 0.2647
Recall@3: 0.5588
Recall@5: 0.7353
MRR:      0.4631


In [22]:
results = {
    'model': 'all-MiniLM-L6-v2 + MultipleNegativesRankingLoss',
    'positive_pairs': len(crosswalk),
    'recall@1': recall_1,
    'recall@3': recall_3,
    'recall@5': recall_5,
    'mrr': mrr,
    'date': str(datetime.date.today())
}
with open(MODEL_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("✅ Evaluation results saved to", MODEL_DIR / 'evaluation_results.json')

✅ Evaluation results saved to ..\models\regmap-embedder\evaluation_results.json
